In [202]:
import pandas as pd 
import altair as alt
alt.data_transformers.enable("vegafusion")


DataTransformerRegistry.enable('vegafusion')

In [203]:
data = pd.read_csv('../data/simpsons_script_lines_cleaned.csv', dtype={'word_count': 'Int64'})

# Q1 - What are the characters that issue more words, and how are the word numbers distributed?

In [204]:
data_q1 = data.groupby('raw_character_text')['word_count'].sum().reset_index()
data_q1

,raw_character_text,word_count
0,"""For Dummies"" Author",9
1,"""Just Stamp the Ticket"" Man",39
2,"""Mario"" #2",8
3,"""Shorts"" Bart",22
4,"""Shorts"" Homer",47
...,...,...
6269,iPod,5
6270,newsstand Operator,7
6271,t,1
6272,teacher,51


In [205]:
top_characters = data_q1.sort_values(by='word_count', ascending=False).head(20)
top_characters
chart = alt.Chart(top_characters).mark_bar().encode(
    x=alt.X('raw_character_text', sort='-y', title='Character'),
    y=alt.Y('word_count', title='Total Word Count'),
    tooltip=['raw_character_text', 'word_count']
).properties(
    title='Top 20 Characters by Total Word Count'
).configure_axisX(
    labelAngle=-45,
    labelFontSize=12
).configure_title(
    fontSize=16,
    anchor='start'
)
chart


alt.Chart(...)

too imbalanced, can't get proper plots yet

In [206]:
#How are the word numbers distributed
chart = alt.Chart(data).mark_bar().encode(
    x=alt.X('word_count', bin=alt.Bin(maxbins=50), title='Word Count'),
    y=alt.Y('count()', title='Frequency'),
    tooltip=['count()']
).properties(
    title='Distribution of Word Counts'
).configure_title(
    fontSize=16,
    anchor='start'
)
chart

alt.Chart(...)

In [207]:
# for each character i want to count the number of unique words they have said
unique_words = data.groupby('raw_character_text')['normalized_text'].apply(lambda x: set(' '.join(x).split())).reset_index().rename(columns={'normalized_text': 'unique_words'})
unique_words['unique_word_count'] = unique_words['unique_words'].apply(len)
data_q1 = data_q1.merge(unique_words[['raw_character_text', 'unique_word_count']], on='raw_character_text', how='left')
data_q1.sort_values(by='word_count', ascending=False,inplace=True)
data_q1


,raw_character_text,word_count,unique_word_count
2680,Homer Simpson,271235,16976
3619,Marge Simpson,124721,10232
613,Bart Simpson,109132,10220
3323,Lisa Simpson,99244,10791
889,C. Montgomery Burns,36212,6372
...,...,...,...
5507,Super Bowl Crowd,1,1
1735,EURO-TRASH VOICE,1,1
5502,Sumo,1,1
6107,Willie's Lawyer,1,1


In [208]:
data_q1_main = data_q1[data_q1['raw_character_text'].isin(['Marge Simpson', 'Homer Simpson', 'Bart Simpson', 'Lisa Simpson', 'C. Montgomery Burns'])]
data_q1_main
images = {
    'Marge Simpson': 'https://img.icons8.com/doodle/48/marge-simpson.png',
    'Homer Simpson': 'https://img.icons8.com/doodle/48/homer-simpson.png',
    'Bart Simpson': 'https://img.icons8.com/doodle/48/bart-simpson.png',
    'Lisa Simpson': 'https://img.icons8.com/doodle/48/lisa-simpson.png',
    'C. Montgomery Burns': 'https://img.icons8.com/doodle/48/charles-montgomery-burns.png',
    'Moe Szyslak': 'https://img.icons8.com/doodle/48/bartender-mo.png'
}

data_q1_main['image'] = data_q1_main['raw_character_text'].map(images)
data_q1_main

/var/folders/95/s4thp5290fd41pw2cj8713k80000gn/T/ipykernel_63326/31924166.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_q1_main['image'] = data_q1_main['raw_character_text'].map(images)


,raw_character_text,word_count,unique_word_count,image
2680,Homer Simpson,271235,16976,https://img.icons8.com/doodle/48/homer-simpson...
3619,Marge Simpson,124721,10232,https://img.icons8.com/doodle/48/marge-simpson...
613,Bart Simpson,109132,10220,https://img.icons8.com/doodle/48/bart-simpson.png
3323,Lisa Simpson,99244,10791,https://img.icons8.com/doodle/48/lisa-simpson.png
889,C. Montgomery Burns,36212,6372,https://img.icons8.com/doodle/48/charles-montg...


* https://iconos8.es/icons/set/simpson

* https://altair-viz.github.io/user_guide/marks/image.html
* https://github.com/vega/altair/issues/1949


In [209]:
plot = alt.Chart(data_q1.head(5)).mark_text().encode(
    x = alt.X('word_count', title='Total Word Count'),
    y = alt.Y('unique_word_count', title='Unique Word Count'),
    text = 'raw_character_text',
    tooltip = ['raw_character_text', 'word_count', 'unique_word_count']
)
plot

alt.Chart(...)

In [210]:
alt.Chart(data_q1_main).mark_image(width=30, height=30).encode(
    x = alt.X('word_count', title='Total Word Count'),
    y = alt.Y('unique_word_count', title='Unique Word Count'),
    url = 'image',
    tooltip = ['raw_character_text', 'word_count', 'unique_word_count']
).properties(
    title='Total Word Count vs Unique Word Count for Main Characters'
).configure_title(
    fontSize=16,
    anchor='start'
)

alt.Chart(...)

In [211]:
x_enc = alt.X(
    "raw_character_text:N",
    sort=data_q1_main["raw_character_text"].tolist(),
    axis=alt.Axis(domain=False, ticks=False, labels=False,title = None)  # amaga labels de text
)

bars = alt.Chart(data_q1_main).mark_bar().encode(
    x=x_enc,
    y=alt.Y("word_count:Q", title="Total Word Count"),
    tooltip=["raw_character_text:N", "word_count:Q"]
)

flags = (
    alt.Chart(data_q1_main)
    .transform_calculate(zero="0")
    .mark_image(width=40, height=40, yOffset=14, clip=False)
    .encode(
        x= x_enc,
        y=alt.Y("zero:Q"),
        url="image:N"
    )
)

(bars + flags).resolve_scale(x="shared").properties(
    title="Total Word Count for Main Characters",
    height=360,
    width=400
)

alt.LayerChart(...)

# Q2 How have the word numbers among the characters evolved throughout the available seasons?

opció 1 -> agrupat per tota una temporada

In [212]:
data_q2_1 = data.groupby(['season','raw_character_text'])['word_count'].sum().reset_index()
data_q2_1

,season,raw_character_text,word_count
0,1,1st Waiter Quartet,9
1,1,2nd Waiter Quartet,14
2,1,3rd Waiter Quartet,8
3,1,4TH QUARTET OF WAITERS,18
4,1,ALL AT THE TABLE,1
...,...,...,...
10472,26,YOUNG JEWISH MAN,27
10473,26,YOUNG MOLEMAN,34
10474,26,Young Grampa,33
10475,26,Young Krusty,7


In [213]:
characters = ['Homer Simpson','Bart Simpson','Marge Simpson','Lisa Simpson','Moe Szyslak']
data_q2_1_main = data_q2_1[data_q2_1['raw_character_text'].isin(characters)]
data_q2_1_main['image'] = data_q2_1_main['raw_character_text'].map(images)
data_q2_1_main

/var/folders/95/s4thp5290fd41pw2cj8713k80000gn/T/ipykernel_63326/2058149966.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_q2_1_main['image'] = data_q2_1_main['raw_character_text'].map(images)


,season,raw_character_text,word_count,image
17,1,Bart Simpson,4868,https://img.icons8.com/doodle/48/bart-simpson.png
85,1,Homer Simpson,8146,https://img.icons8.com/doodle/48/homer-simpson...
111,1,Lisa Simpson,1871,https://img.icons8.com/doodle/48/lisa-simpson.png
125,1,Marge Simpson,3662,https://img.icons8.com/doodle/48/marge-simpson...
135,1,Moe Szyslak,444,https://img.icons8.com/doodle/48/bartender-mo.png
...,...,...,...,...
10227,26,Bart Simpson,2048,https://img.icons8.com/doodle/48/bart-simpson.png
10302,26,Homer Simpson,7550,https://img.icons8.com/doodle/48/homer-simpson...
10338,26,Lisa Simpson,2042,https://img.icons8.com/doodle/48/lisa-simpson.png
10356,26,Marge Simpson,3321,https://img.icons8.com/doodle/48/marge-simpson...


In [214]:
points = alt.Chart(data_q2_1_main).mark_point(filled = True).encode(
    x=alt.X('season:O', title='Season'),
    y=alt.Y('word_count:Q', title='Total Word Count'),
    color=alt.Color('raw_character_text:N', title='Character'),
    tooltip=['season', 'raw_character_text', 'word_count']
)

lines = alt.Chart(data_q2_1_main).mark_line().encode(
    x=alt.X('season:O', title='Season'),
    y=alt.Y('word_count:Q', title='Total Word Count'),
    color=alt.Color('raw_character_text:N', title='Character')
    
)

(points + lines).properties(
    title='Total Word Count per Season for Main Characters'
).configure_title(
    fontSize=16,
    anchor='start'
).configure_axisX(
    labelAngle=-0,
)

alt.LayerChart(...)

In [215]:
points_faces = alt.Chart(data_q2_1_main).mark_image(width=20, height=20).encode(
x=alt.X('season:O', title='Season'),
y=alt.Y('word_count:Q', title='Total Word Count'),
url='image:N',
tooltip=['season:O', 'raw_character_text:N', 'word_count:Q']
)

lines = alt.Chart(data_q2_1_main).mark_line(strokeWidth=2).encode(
x=alt.X('season:O', title='Season'),
y=alt.Y('word_count:Q', title='Total Word Count'),
color=alt.Color('raw_character_text:N', title='Character', legend=None),
detail='raw_character_text:N'
)

(lines + points_faces).properties(
title='Total Word Count per Season for Main Characters'
).configure_title(
fontSize=16,
anchor='start'
).configure_axisX(
labelAngle=0
)



alt.LayerChart(...)

opcio 2 agrupat per cada capitol

In [216]:
data

,id,season,number_in_season,episode_id,line_number_in_episode,character_id,raw_character_text,normalized_text,word_count
0,9549,2,19,32,209,464.0,Miss Hoover,no actually it was a little of both sometimes ...,31
1,9550,2,19,32,210,9.0,Lisa Simpson,wheres mr bergstrom,3
2,9551,2,19,32,211,464.0,Miss Hoover,i dont know although id sure like to talk to h...,22
3,9552,2,19,32,212,9.0,Lisa Simpson,that life is worth living,5
4,9553,2,19,32,213,40.0,Edna Krabappel-Flanders,the polls will be open from now until the end ...,33
...,...,...,...,...,...,...,...,...,...
132016,9544,2,19,32,204,464.0,Miss Hoover,im back,2
132017,9545,2,19,32,205,464.0,Miss Hoover,you see class my lyme disease turned out to be,10
132018,9546,2,19,32,206,464.0,Miss Hoover,psy-cho-so-ma-tic,1
132019,9547,2,19,32,207,119.0,Ralph Wiggum,does that mean you were crazy,6


In [217]:
data_q2_2 = data.groupby(['season', 'episode_id', 'raw_character_text'])['word_count'].sum().reset_index()
data_q2_2

,season,episode_id,raw_character_text,word_count
0,1,1,Announcer,131
1,1,1,Barney Gumble,114
2,1,1,Bart Simpson,335
3,1,1,Bubbles,17
4,1,1,C. Montgomery Burns,45
...,...,...,...,...
20158,26,568,Ned Flanders,82
20159,26,568,Rev. Timothy Lovejoy,109
20160,26,568,SECURITY THUG #1,25
20161,26,568,SECURITY THUG #2,33


In [220]:
data_q2_2_main = data_q2_2[data_q2_2['raw_character_text'].isin(characters)]
data_q2_2_main['image'] = data_q2_2_main['raw_character_text'].map(images)
data_q2_2_main = data_q2_2_main.merge(data[['episode_id','number_in_season']].drop_duplicates(),how='left', on='episode_id')
data_q2_2_main

/var/folders/95/s4thp5290fd41pw2cj8713k80000gn/T/ipykernel_63326/4292786065.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_q2_2_main['image'] = data_q2_2_main['raw_character_text'].map(images)


,season,episode_id,raw_character_text,word_count,image,number_in_season
0,1,1,Bart Simpson,335,https://img.icons8.com/doodle/48/bart-simpson.png,1
1,1,1,Homer Simpson,900,https://img.icons8.com/doodle/48/homer-simpson...,1
2,1,1,Lisa Simpson,222,https://img.icons8.com/doodle/48/lisa-simpson.png,1
3,1,1,Marge Simpson,373,https://img.icons8.com/doodle/48/marge-simpson...,1
4,1,1,Moe Szyslak,26,https://img.icons8.com/doodle/48/bartender-mo.png,1
...,...,...,...,...,...,...
2622,26,567,Moe Szyslak,589,https://img.icons8.com/doodle/48/bartender-mo.png,15
2623,26,568,Bart Simpson,66,https://img.icons8.com/doodle/48/bart-simpson.png,16
2624,26,568,Homer Simpson,310,https://img.icons8.com/doodle/48/homer-simpson...,16
2625,26,568,Lisa Simpson,94,https://img.icons8.com/doodle/48/lisa-simpson.png,16


In [228]:
points = alt.Chart(data_q2_2_main).mark_point(filled = True, size=60).encode(
    x=alt.X('episode_id:O', title='Season'),
    y=alt.Y('word_count:Q', title='Total Word Count'),
    color=alt.Color('raw_character_text:N', title='Character'),
    tooltip=['season', 'number_in_season','raw_character_text', 'word_count']
)

lines = alt.Chart(data_q2_2_main).mark_line().encode(
    x=alt.X('episode_id:O', title='Season'),
    y=alt.Y('word_count:Q', title='Total Word Count'),
    color=alt.Color('raw_character_text:N', title='Character')
    
)

(points + lines).properties(
    title='Total Word Count per Season for Main Characters',
    width=2000,
).configure_title(
    fontSize=16,
    anchor='start'
).configure_axisX(
    labelAngle=0,
)

alt.LayerChart(...)

# Q3 I would like to compare the word distribution for a pair of concrete characters, for a selected season, and, in addition, for a selected episode.

Assumim una sola season

In [233]:
data_q3 = data.groupby(['season', 'number_in_season', 'raw_character_text'])['word_count'].sum().reset_index()
data_q3['image'] = data_q3['raw_character_text'].map(images)
characters = ['Bart Simpson', 'Lisa Simpson']
data_q3 = data_q3[data_q3['season'] == 1]
data_q3 = data_q3[data_q3['raw_character_text'].isin(characters)]
data_q3


,season,number_in_season,raw_character_text,word_count,image
2,1,1,Bart Simpson,335,https://img.icons8.com/doodle/48/bart-simpson.png
18,1,1,Lisa Simpson,222,https://img.icons8.com/doodle/48/lisa-simpson.png
37,1,2,Bart Simpson,619,https://img.icons8.com/doodle/48/bart-simpson.png
48,1,2,Lisa Simpson,71,https://img.icons8.com/doodle/48/lisa-simpson.png
58,1,3,Bart Simpson,209,https://img.icons8.com/doodle/48/bart-simpson.png
71,1,3,Lisa Simpson,54,https://img.icons8.com/doodle/48/lisa-simpson.png
94,1,4,Bart Simpson,189,https://img.icons8.com/doodle/48/bart-simpson.png
100,1,4,Lisa Simpson,150,https://img.icons8.com/doodle/48/lisa-simpson.png
126,1,5,Bart Simpson,795,https://img.icons8.com/doodle/48/bart-simpson.png
134,1,5,Lisa Simpson,224,https://img.icons8.com/doodle/48/lisa-simpson.png


based on the code here : https://altair-viz.github.io/gallery/ranged_dot_plot.html

In [255]:
chart = (
    alt.Chart(data_q3)
    .transform_filter(
        alt.FieldOneOfPredicate(
            field="raw_character_text",
            oneOf=["Bart Simpson", "Lisa Simpson"]
        )
    )
    .encode(
        x=alt.X("word_count:Q", title="Word Count"),
        y=alt.Y("number_in_season:O", title="Episode Number in Season"),
        detail="number_in_season:O",
        tooltip=["season:O", "number_in_season:O", "raw_character_text:N", "word_count:Q"]
    )
)

line = chart.mark_line(color="#db646f", strokeWidth=4)

faces = chart.mark_image(width=20, height=20).encode(
    url="image:N"
)

(line + faces).properties(
    title="Bart vs Lisa Word Count by Episode (Season 1)"
)

alt.LayerChart(...)